# Final Comparison — Phase 4 Rollup (v2 pipeline + Phases 1–3)

Consolidates:
1. Classical baselines (Gaussian / Isolation Forest / VAE) from `baseline_comparison.pkl`
2. Ablation: VAE alone → Phase-1 FPR-targeted adaptive → Phase-2 full model
3. Phase-3 multi-scale ensemble vs v1/v2 alone
4. Honest v1 (W=30) vs v2-improved comparison against success targets

**Success targets** (from the improvement plan):
| Metric | ID | Drift (good) | Drift (stretch) |
|---|---|---|---|
| PR-AUC | ≥0.90 | ≥0.35 | ≥0.42 |
| F1 | ≥0.90 | ≥0.28 | ≥0.40 |
| Precision | ≥0.95 | ≥0.20 | ≥0.30 |
| Recall | ≥0.85 | 0.65–0.80 | ≥0.70 |
| FPR | ≤0.001 | ≤0.03 | ≤0.02 |


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, copy, time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from scipy import stats
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

def resolve_base():
    here = Path.cwd().resolve()
    candidates = [
        here if here.name == 'module3' else None,
        here.parent if here.name == 'module3_pipeline_v2' else None,
        Path(r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'),
        Path(r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'),
    ]
    for c in candidates:
        if c is not None and (c / 'models_v2').exists():
            return str(c)
    raise FileNotFoundError('Could not locate module3 BASE (models_v2 missing).')

BASE = resolve_base()
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
WIN_DIR_V1 = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models_v2')
MODEL_DIR_V1 = os.path.join(BASE, 'models')
print('BASE =', BASE)

ALL_SETS = ['cc1_test', 'drift_cc2']
TARGETS = {
    'id': {'auc_pr': 0.90, 'f1': 0.90, 'precision': 0.95, 'recall': 0.85, 'fpr': 0.001},
    'drift_good': {'auc_pr': 0.35, 'f1': 0.28, 'precision': 0.20, 'recall': 0.65, 'fpr': 0.03},
    'drift_stretch': {'auc_pr': 0.42, 'f1': 0.40, 'precision': 0.30, 'recall': 0.70, 'fpr': 0.02},
}

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

vae_eval = load_pkl(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'))
adaptive_eval = load_pkl(os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl'))
incremental_eval = load_pkl(os.path.join(MODEL_DIR, 'incremental_learning_eval.pkl'))
baseline_path = os.path.join(MODEL_DIR, 'baseline_comparison.pkl')
baseline_eval = load_pkl(baseline_path) if os.path.exists(baseline_path) else None
phase3_path = os.path.join(MODEL_DIR, 'phase3_multiscale_eval.pkl')
phase3_eval = load_pkl(phase3_path) if os.path.exists(phase3_path) else None
v1_final_path = os.path.join(MODEL_DIR_V1, 'final_comparison_results.pkl')
v1_final = load_pkl(v1_final_path) if os.path.exists(v1_final_path) else None
print('Artifacts loaded.')
print('  phase1 threshold_mode:', adaptive_eval.get('threshold_mode'))
print('  phase2 n_finetunes:', incremental_eval.get('incremental_learning', {}).get('n_finetunes'))
print('  phase3 present:', phase3_eval is not None)
print('  v1 final present:', v1_final is not None)


## Section 1 — Baseline comparison (v2)


In [ ]:
if baseline_eval is None:
    print('baseline_comparison.pkl missing — skip Section 1')
else:
    # Support both nested layouts
    methods = baseline_eval if 'gaussian' in baseline_eval else baseline_eval.get('methods', baseline_eval)
    print(f'{"set":12s} {"method":18s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"P":>7s} {"R":>7s}')
    for name in ALL_SETS:
        for method in ['gaussian', 'isolation_forest', 'vae']:
            if method not in methods or name not in methods[method]:
                continue
            r = methods[method][name]
            print(f'{name:12s} {method:18s} {r.get("auc_pr", float("nan")):8.4f} '
                  f'{r.get("auc_roc", float("nan")):9.4f} {r.get("f1", float("nan")):7.3f} '
                  f'{r.get("precision", float("nan")):7.3f} {r.get("recall", float("nan")):7.3f}')
        print()


## Section 2 — Ablation: VAE alone → Phase-1 adaptive → Phase-2 full


In [ ]:
ablation = {'vae_alone': {}, 'phase1_adaptive': {}, 'phase2_full': {}}
for name in ALL_SETS:
    ablation['vae_alone'][name] = {
        'auc_pr': vae_eval['auc'][name]['auc_pr'],
        'auc_roc': vae_eval['auc'][name]['auc_roc'],
        'f1': vae_eval['precision_recall'][name]['val_p99']['f1'],
        'precision': vae_eval['precision_recall'][name]['val_p99']['precision'],
        'recall': vae_eval['precision_recall'][name]['val_p99']['recall'],
        'fpr': vae_eval['precision_recall'][name]['val_p99'].get('fpr', float('nan')),
    }
    ar = adaptive_eval['results'][name]
    ablation['phase1_adaptive'][name] = {
        'auc_pr': vae_eval['auc'][name]['auc_pr'],  # thresholding does not change ranking
        'auc_roc': vae_eval['auc'][name]['auc_roc'],
        'f1': ar['f1'], 'precision': ar['precision'], 'recall': ar['recall'],
        'fpr': ar.get('fpr', float('nan')),
    }

# Full model: drift uses Phase-2 treatment; ID uses Phase-1 adaptive (IL no-op in-distribution)
ablation['phase2_full']['cc1_test'] = dict(ablation['phase1_adaptive']['cc1_test'])
treat = incremental_eval['incremental_learning']['treatment']
ablation['phase2_full']['drift_cc2'] = {
    'auc_pr': vae_eval['auc']['drift_cc2']['auc_pr'],  # approx; IL can move scores slightly
    'auc_roc': vae_eval['auc']['drift_cc2']['auc_roc'],
    'f1': treat['f1'], 'precision': treat['precision'], 'recall': treat['recall'],
    'fpr': treat.get('fpr', float('nan')),
}

print(f'{"set":12s} {"config":18s} {"PR-AUC":>8s} {"F1":>7s} {"P":>7s} {"R":>7s} {"FPR":>8s}')
for name in ALL_SETS:
    for cfg in ['vae_alone', 'phase1_adaptive', 'phase2_full']:
        m = ablation[cfg][name]
        print(f'{name:12s} {cfg:18s} {m["auc_pr"]:8.4f} {m["f1"]:7.3f} {m["precision"]:7.3f} '
              f'{m["recall"]:7.3f} {m.get("fpr", float("nan")):8.4f}')
    print()


## Section 3 — Phase-3 multi-scale vs v1/v2 alone


In [ ]:
if phase3_eval is None:
    print('phase3_multiscale_eval.pkl missing — run multiscale_comparison.ipynb first')
    phase3_summary = None
else:
    best = phase3_eval['best_rule']
    phase3_summary = {
        'best_rule': best,
        'id_gate_passed': phase3_eval['id_gate_passed'],
        'cc1_test': phase3_eval['results']['cc1_test'][best],
        'drift_cc2': phase3_eval['results']['drift_cc2'][best],
        'v2_alone_cc1': phase3_eval['results']['cc1_test']['v2_alone'],
        'v2_alone_drift': phase3_eval['results']['drift_cc2']['v2_alone'],
        'v1_alone_cc1': phase3_eval['results']['cc1_test']['v1_alone'],
        'v1_alone_drift': phase3_eval['results']['drift_cc2']['v1_alone'],
    }
    print(f'Best rule: {best}  ID_gate={phase3_eval["id_gate_passed"]}')
    print(f'{"set":12s} {"method":14s} {"PR-AUC":>8s} {"F1":>7s} {"P":>7s} {"R":>7s} {"FPR":>8s}')
    for name in ALL_SETS:
        for method in ['v1_alone', 'v2_alone', best]:
            r = phase3_eval['results'][name][method]
            print(f'{name:12s} {method:14s} {r["auc_pr"]:8.4f} {r["f1"]:7.3f} {r["precision"]:7.3f} '
                  f'{r["recall"]:7.3f} {r["fpr"]:8.4f}')
        print()


## Section 4 — Target checklist + v1 vs improved-v2


In [ ]:
def check_targets(tag, metrics, target):
    rows = []
    for k, thr in target.items():
        val = metrics.get(k, float('nan'))
        if k == 'fpr':
            ok = val <= thr
        elif k == 'recall':
            ok = val >= thr
        else:
            ok = val >= thr
        rows.append((k, val, thr, ok))
        print(f'  {tag:8s} {k:10s}={val:.4f}  target={thr}  {"PASS" if ok else "FAIL"}')
    return all(ok for *_, ok in rows)

print('=== Recommended operating point for thesis reporting ===')
# Prefer Phase-3 best if it passed ID gate; else Phase-2 full
if phase3_summary and phase3_summary['id_gate_passed']:
    chosen_name = f"phase3:{phase3_summary['best_rule']}"
    chosen = {
        'cc1_test': phase3_summary['cc1_test'],
        'drift_cc2': phase3_summary['drift_cc2'],
    }
else:
    chosen_name = 'phase2_full'
    chosen = {
        'cc1_test': ablation['phase2_full']['cc1_test'],
        'drift_cc2': ablation['phase2_full']['drift_cc2'],
    }
print('Chosen configuration:', chosen_name)

print('\\nID targets:')
id_pass = check_targets('ID', {
    'auc_pr': chosen['cc1_test'].get('auc_pr', chosen['cc1_test'].get('auc_pr', float('nan'))),
    'f1': chosen['cc1_test']['f1'],
    'precision': chosen['cc1_test']['precision'],
    'recall': chosen['cc1_test']['recall'],
    'fpr': chosen['cc1_test'].get('fpr', float('nan')),
}, TARGETS['id'])

print('\\nDrift GOOD targets:')
drift_good = check_targets('DRIFT', chosen['drift_cc2'], TARGETS['drift_good'])
print('\\nDrift STRETCH targets:')
drift_stretch = check_targets('DRIFT', chosen['drift_cc2'], TARGETS['drift_stretch'])

# v1 comparison if available
if v1_final is not None and 'ablation_study' in v1_final:
    print('\\n=== v1 Full Model vs chosen ===')
    v1_abl = v1_final['ablation_study']
    print(f'{"set":12s} {"pipeline":22s} {"PR-AUC":>8s} {"F1":>7s} {"P":>7s} {"R":>7s}')
    for name in ALL_SETS:
        v1m = v1_abl['full_model'][name]
        cm = chosen[name]
        print(f'{name:12s} {"v1 full (W=30)":22s} {v1m.get("auc_pr", float("nan")):8.4f} '
              f'{v1m["f1"]:7.3f} {v1m["precision"]:7.3f} {v1m["recall"]:7.3f}')
        print(f'{name:12s} {chosen_name[:22]:22s} {cm.get("auc_pr", float("nan")):8.4f} '
              f'{cm["f1"]:7.3f} {cm["precision"]:7.3f} {cm["recall"]:7.3f}')
        print()


## Section 5 — Plots


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
configs = ['vae_alone', 'phase1_adaptive', 'phase2_full']
labels = ['VAE alone', 'Phase1 adaptive', 'Phase2 full']
for ax, name, title in zip(axes, ALL_SETS, ['In-distribution (cc1_test)', 'Drift (drift_cc2)']):
    f1s = [ablation[c][name]['f1'] for c in configs]
    fprs = [ablation[c][name].get('fpr', np.nan) for c in configs]
    x = np.arange(len(configs))
    ax.bar(x - 0.2, f1s, 0.4, label='F1')
    ax.bar(x + 0.2, fprs, 0.4, label='FPR')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
    ax.set_title(title); ax.legend(); ax.set_ylim(0, 1.05)
    ax.axhline(0.90 if name == 'cc1_test' else 0.28, color='gray', ls='--', lw=0.8)
plt.tight_layout()
fig_path = os.path.join(MODEL_DIR, 'phase4_ablation_f1_fpr.png')
plt.savefig(fig_path, dpi=120); plt.show()
print('Saved', fig_path)

if phase3_summary:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    methods = ['v1_alone', 'v2_alone', phase3_summary['best_rule']]
    x = np.arange(len(methods)); width = 0.35
    id_f1 = [phase3_eval['results']['cc1_test'][m]['f1'] for m in methods]
    dr_f1 = [phase3_eval['results']['drift_cc2'][m]['f1'] for m in methods]
    ax.bar(x - width/2, id_f1, width, label='cc1_test F1')
    ax.bar(x + width/2, dr_f1, width, label='drift_cc2 F1')
    ax.set_xticks(x); ax.set_xticklabels(methods)
    ax.set_ylabel('F1'); ax.set_title('Phase 3 multi-scale vs single-window'); ax.legend()
    plt.tight_layout()
    fig_path2 = os.path.join(MODEL_DIR, 'phase4_multiscale_f1.png')
    plt.savefig(fig_path2, dpi=120); plt.show()
    print('Saved', fig_path2)


## Save final rollup


In [ ]:
save_results = {
    'phase': 4,
    'targets': TARGETS,
    'baseline_comparison': baseline_eval,
    'ablation_study': ablation,
    'phase3': phase3_summary,
    'chosen_configuration': chosen_name,
    'chosen_metrics': chosen,
    'target_checks': {
        'id_pass': bool(id_pass),
        'drift_good_pass': bool(drift_good),
        'drift_stretch_pass': bool(drift_stretch),
    },
}
if v1_final is not None and 'ablation_study' in v1_final:
    save_results['v1_vs_chosen'] = {
        name: {
            'v1_full': v1_final['ablation_study']['full_model'][name],
            'chosen': chosen[name],
        } for name in ALL_SETS
    }

out_path = os.path.join(MODEL_DIR, 'final_comparison_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

phase4_path = os.path.join(MODEL_DIR, 'phase4_final_rollup.pkl')
with open(phase4_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {phase4_path}')


## How to read this

1. **Section 2** shows whether Phase-1/2 improved drift FPR/Precision without breaking ID.
2. **Section 3** shows whether multi-scale broke the W=30 vs W=60 Pareto tradeoff.
3. **Section 4** is the thesis checklist against the pre-registered targets.
4. Prefer reporting the **chosen_configuration** named in the saved pickle — do not mix
   operating points across phases when quoting a single "final" number.
